# MediCare AI Backend on Google Colab

Run the full FastAPI backend with **PaddleOCR** on Google Colab's free tier.

## Before you start

1. Make sure your project code is on **GitHub** or in **Google Drive**.
2. Add the following secrets to Colab (click the key icon on the left sidebar):
   - `MEDICARE_JWT_SECRET`
   - `MEDICARE_AI_API_KEY`
   - `SUPABASE_SERVICE_ROLE_KEY`
   - `NGROK_AUTH_TOKEN` (get it from https://dashboard.ngrok.com/get-started/your-authtoken)

## Limitations

- Free Colab disconnects after ~90 minutes of inactivity.
- Each ngrok session gives a new public URL, so you must update Vercel each time.
- Free Colab uses CPU only; OCR is slower than on a GPU.


## Step 1: Load the project code

Choose **Option A** (GitHub) or **Option B** (Google Drive) and run only that cell.

In [ ]:
# Option A: Clone from GitHub (recommended)
# Replace YOUR_USERNAME with your GitHub username and adjust the repo name if needed.
!git clone https://github.com/YOUR_USERNAME/medicare-ai.git
%cd medicare-ai

In [ ]:
# Option B: Use Google Drive
# Upload the medicare-ai folder to your Google Drive, then run this cell.
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/medicare-ai

## Step 2: Install dependencies

This installs FastAPI, PaddlePaddle, PaddleOCR, PyMuPDF, Supabase, and all other backend dependencies. It may take a few minutes.

In [ ]:
!pip install -q -r requirements.txt

## Step 3: Load secrets and configure the environment

Secrets are read from Colab's secret store so they are never hardcoded in this notebook.

In [ ]:
from google.colab import userdata
import os

os.environ['MEDICARE_ENV'] = 'production'
os.environ['MEDICARE_AI_PROVIDER'] = 'groq'
os.environ['MEDICARE_AI_MODEL'] = 'openai/gpt-oss-20b'
os.environ['MEDICARE_UPLOAD_DIR'] = '/tmp/uploads'

os.environ['MEDICARE_JWT_SECRET'] = userdata.get('MEDICARE_JWT_SECRET')
os.environ['MEDICARE_AI_API_KEY'] = userdata.get('MEDICARE_AI_API_KEY')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = userdata.get('SUPABASE_SERVICE_ROLE_KEY')

os.environ['NEXT_PUBLIC_SUPABASE_URL'] = 'https://fftwxbmoacdxcquowqzv.supabase.co'
os.environ['NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY'] = 'sb_publishable_vKrzdw4n94QPVKwAtfDwOQ_w7oG_LPY'

# Allow your Vercel frontend domains. Add preview URLs here if needed.
os.environ['CORS_ALLOWED_ORIGINS'] = (
    'https://medicare-ai.vercel.app,'
    'https://medicare-ai-sigma.vercel.app,'
    'https://medicare-ai-ali-raza4.vercel.app,'
    'http://localhost:3000'
)

print('Environment configured.')

## Step 4: Start the FastAPI backend

The backend runs on port `8000`. The first call that uses PaddleOCR will be slower because models are downloaded/initialized on demand.

In [ ]:
import subprocess
import time

backend_process = subprocess.Popen(
    ['uvicorn', 'backend.app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to start
time.sleep(8)
print('Backend started with PID:', backend_process.pid)

## Step 5: Verify the backend is healthy

In [ ]:
import requests

response = requests.get('http://localhost:8000/api/health')
print('Status:', response.status_code)
print('Response:', response.json())

## Step 6: Create a public tunnel with ngrok

This exposes your local Colab backend to the internet so the Vercel frontend can reach it.

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))
public_url = ngrok.connect(8000, 'http')

print('Public backend URL:', public_url)
print('Copy this URL and set it as NEXT_PUBLIC_API_BASE_URL in Vercel.')

## Step 7: Point Vercel to Colab

1. Go to your Vercel project dashboard.
2. Open **Settings → Environment Variables**.
3. Add `NEXT_PUBLIC_API_BASE_URL` with the ngrok URL printed above.
4. Apply it to **Production**, **Preview**, and **Development**.
5. Redeploy the frontend.

Now upload a medical image from the deployed frontend. The file will be processed by PaddleOCR running on Colab.

## Step 8: Keep the session alive (optional)

Run this cell in a separate tab to reduce the chance of Colab disconnecting due to inactivity. It will not prevent the eventual 12-hour runtime limit on free Colab.

In [ ]:
import time

print('Keeping session alive. Press the stop button to cancel.')
while True:
    time.sleep(60)